# Shared Parkinson dataset for every Colab model

Run the next cell at the start of any model notebook, or copy it into that notebook. It mounts Drive and loads the same versioned dataset for all architectures. Select a GPU kernel when your training requires one; loading the data does not require a GPU.

The dataset is already stored at **My Drive / ParkinsonResearch / datasets / parkinson_dataset_v1.zip**. It contains 147 original PNGs, 147 trace TXT files and 20 annotation/workbook/audit files. Original bytes, labels, subject assignments and eligibility flags are preserved. There are 103 eligible development images and 36 eligible locked-test images. Other images remain present with their existing exclusion/quarantine flags.

Authorize the same Google account containing this folder. The cell copies the ZIP to Colab's temporary disk, verifies its checksum, extracts it, and verifies every dataset file. Rerun it after a runtime reset. This reads the existing Drive copy; there is no need to upload again for each model.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import shutil
import zipfile

drive.mount('/content/drive')
DRIVE_PROJECT = Path('/content/drive/MyDrive/ParkinsonResearch')
SOURCE_ZIP = DRIVE_PROJECT / 'datasets/parkinson_dataset_v1.zip'
DATA_ROOT = Path('/content/ParkinsonData')
DATASET_DIR = DATA_ROOT / 'datasets'
MANIFEST = DATASET_DIR / 'annotations/fold_manifest.csv'
RUNS_DIR = DRIVE_PROJECT / 'runs'
EXPECTED_SHA256 = '85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6'

if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f'Shared dataset ZIP is missing: {SOURCE_ZIP}')
local_zip = Path('/content/parkinson_dataset_v1.zip')
shutil.copy2(SOURCE_ZIP, local_zip)
if hashlib.sha256(local_zip.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise ValueError('Dataset ZIP differs from version 1. Check the file or update the version explicitly.')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(local_zip) as archive:
    for name in archive.namelist():
        if not (DATA_ROOT / name).resolve().is_relative_to(DATA_ROOT.resolve()):
            raise ValueError(f'Unsafe archive path: {name}')
    archive.extractall(DATA_ROOT)
metadata = json.loads((DATA_ROOT / 'dataset_package.json').read_text())
for name, info in metadata['files'].items():
    if not (DATA_ROOT / name).resolve().is_relative_to(DATA_ROOT.resolve()):
        raise ValueError(f'Unsafe dataset path: {name}')
    if hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest() != info['sha256']:
        raise ValueError(f'Dataset file failed verification: {name}')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset:', metadata['dataset_version'])
print('Images:', metadata['images'], '| Trace files:', metadata['traces'])
print('Dataset directory:', DATASET_DIR)
print('Fold manifest:', MANIFEST)
print('Save each model under:', RUNS_DIR)


## Use it in any training notebook

Your model code and dependencies are loaded separately. Every model can use these variables:

- `DATASET_DIR`: images, traces and annotations (`/content/ParkinsonData/datasets`).
- `MANIFEST`: the common `fold_manifest.csv` with the prescribed subject splits and finalized labels.
- `DATA_ROOT`: the parent of `datasets`; resolve relative paths from the CSVs against this directory.
- `RUNS_DIR`: persistent Drive storage for checkpoints and reports; create a separate subfolder per model/recipe.

For the current project's training scripts, pass `--manifest str(MANIFEST)` and `--root str(DATA_ROOT)`. For example, in an existing command list:

```python
command += ['--manifest', str(MANIFEST), '--root', str(DATA_ROOT)]
```

In the baseline notebook, replace its `MANIFEST = ...` setting with the shared `MANIFEST` above and change the `--root` value in `COMMON` from `PROJECT_DIR` to `DATA_ROOT`. Keep `PROJECT_DIR` pointing to the model-code directory. Choose a new Drive output folder for this full-dataset manifest instead of reusing checkpoints from the previous development-only export.

Use rows for the requested development fold and training/validation role; preserve `cv_eligible`. The locked test set remains for final evaluation after model selection. Trace-to-image pairing status is recorded in annotations; a matching pair is not assumed.

Examples of persistent output folders: `RUNS_DIR / 'wideresnet_ce_v1'`, `RUNS_DIR / 'wideresnet_focal_v1'`, `RUNS_DIR / 'metric_learning_v1'`, and `RUNS_DIR / 'rbf_v1'`.
